# FSAI, Inc. — DIA (Disaster Intelligence Atlas)
## Phase 1 (CRISP-DM): Data Understanding & Initial Preprocessing Pipeline
**Study area:** Zambales province + Olongapo City (Region III)

This notebook is the reproducible preprocessing workflow for Phase 1. It:

1. Downloads the **original** datasets straight from their sources (nothing is typed in by hand):
   - **Project NOAH hazard maps** (flood 5/25/100-yr, storm surge SSA 1–4): Hugging Face mirror `bettergovph/project-noah-hazard-maps` (ODbL)
   - **NASA POWER** daily rainfall, temperature, and humidity (1981–present), one point per LGU
   - **geoBoundaries** (gbOpen) Philippine province (ADM2) and municipality/city (ADM3) boundaries
2. Inspects and documents each dataset's initial condition (EDA on the raw data).
3. Cleans, imputes, integrates, transforms, and reduces the data.
4. Runs the Phase 1 hypothesis tests.
5. Exports cleaned datasets, figures, a data dictionary, and `run_summary.json` (every number the report quotes).

**How to run:** In Colab, choose *Runtime → Run all*. The first run downloads a few hundred MB and takes about 10–25 minutes. When it finishes, the last cell zips `outputs/` for download.

> Nothing in this notebook generates or simulates data. If a source can't be reached, the cell fails loudly instead of substituting values.

## 0. Setup

In [ ]:
# Colab already ships pandas/numpy/matplotlib/seaborn/geopandas; install anything missing.
import importlib, subprocess, sys
for pkg, mod in [("geopandas", "geopandas"), ("pyogrio", "pyogrio"), ("huggingface_hub", "huggingface_hub"),
                 ("scipy", "scipy"), ("seaborn", "seaborn")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [ ]:
import os, re, json, time, zipfile, glob, shutil, warnings, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from scipy import stats
from shapely.validation import make_valid
import pyogrio
# geoBoundaries PHL ADM3 is one huge GeoJSON; GDAL's default 200 MB per-object cap rejects it. 0 = no cap.
os.environ["OGR_GEOJSON_MAX_OBJ_SIZE"] = "0"
pyogrio.set_gdal_config_options({"OGR_GEOJSON_MAX_OBJ_SIZE": "0"})

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 50)
print("geopandas", gpd.__version__, "| pandas", pd.__version__)

### Configuration
Every tunable choice lives here, so the methodology section of the report can cite exact values.

In [ ]:
# ---- Sources ---------------------------------------------------------------
HF_REPO_ID      = "bettergovph/project-noah-hazard-maps"
AOI_KEYWORDS    = ["zambales", "olongapo"]          # matched against NOAH province file names
RETURN_PERIODS  = ["5yr", "25yr", "100yr"]
INCLUDE_STORM_SURGE = True                            # Zambales is coastal (Subic Bay, west coast)
GB_API          = "https://www.geoboundaries.org/api/current/gbOpen/PHL/{lvl}/"
POWER_URL       = "https://power.larc.nasa.gov/api/temporal/daily/point"
POWER_PARAMS    = ["PRECTOTCORR", "T2M", "RH2M"]      # rainfall mm/day, air temp °C, rel. humidity %
POWER_START     = "19810101"                          # earliest PRECTOTCORR day in POWER daily API
POWER_END       = (dt.date.today() - dt.timedelta(days=14)).strftime("%Y%m%d")  # POWER has a processing lag

# Known Zambales LGUs (PSA PSGC) + Olongapo City (HUC). Used only to cross-check the spatial AOI selection.
EXPECTED_LGUS = ["Botolan","Cabangan","Candelaria","Castillejos","Iba","Masinloc","Olongapo","Palauig",
                 "San Antonio","San Felipe","San Marcelino","San Narciso","Santa Cruz","Subic"]

# ---- Processing choices ----------------------------------------------------
METRIC_CRS        = "EPSG:32651"  # WGS 84 / UTM 51N — metric CRS for areas (scale error <0.1% at 120°E)
WEB_CRS           = "EPSG:4326"   # what MapLibre/Leaflet expect for GeoJSON
SLIVER_M2         = 1.0           # polygons smaller than this are digitising artefacts
SIMPLIFY_TOL_M    = 5.0           # web-layer simplification tolerance (metres)
WET_DAY_MM        = 1.0           # WMO/ETCCDI wet-day threshold
MAX_INTERP_DAYS   = 3             # longest gap filled for smooth variables (T2M, RH2M)
RAIN_MAX_PLAUSIBLE= 1000.0        # mm/day. Above this is flagged; the PH 24-h record is ~1,085 mm (Baguio, 1911)
CORR_DROP         = 0.95          # |r| above this → redundant feature
WET_SEASON_MONTHS = [5,6,7,8,9,10]  # Coronas Type I climate (Zambales): dry Nov–Apr, wet May–Oct
LOCAL_DATA_DIR    = None          # optional: folder with pre-downloaded raw files (same layout as data/raw)

# ---- Paths -----------------------------------------------------------------
ROOT = Path(".").resolve()
RAW, INTERIM, OUT = ROOT/"data/raw", ROOT/"data/interim", ROOT/"outputs"
FIG, CLEAN = OUT/"figures", OUT/"clean"
for p in [RAW, INTERIM, OUT, FIG, CLEAN]: p.mkdir(parents=True, exist_ok=True)

SUMMARY = {"run_utc": dt.datetime.utcnow().isoformat(timespec="seconds")+"Z",
           "config": {k: v for k, v in dict(globals()).items()
                      if k.isupper() and isinstance(v, (str, int, float, list, bool, type(None)))}}

# ---- Plot style (ordinal hazard/return-period ramps = one hue, light→dark) --
SEQ3   = ["#86b6ef", "#2a78d6", "#104281"]            # Low / Medium / High  and  5 / 25 / 100-yr
CAT    = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
HAZ_LABEL = {1: "Low", 2: "Medium", 3: "High"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 160, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e3",
                     "grid.linewidth": 0.6, "axes.edgecolor": "#8a8984", "font.size": 10,
                     "axes.titleweight": "bold", "axes.titlesize": 11})

def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/f"{name}.png", bbox_inches="tight"); plt.show(); plt.close()

def put(key, value):
    # Store a value in SUMMARY (numpy types → python) and echo it.
    def conv(o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
        if isinstance(o, dict): return {str(k): conv(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [conv(v) for v in o]
        if isinstance(o, pd.Timestamp): return o.isoformat()
        return o
    SUMMARY[key] = conv(value); print(f"[summary] {key} = {SUMMARY[key] if len(str(SUMMARY[key]))<400 else '…'}")

## 1. Data collection
### 1.1 Project NOAH hazard maps (Hugging Face mirror)
Step one is listing the repository, so the file names below are the **actual** ones, not guesses. Each province is one ZIP of ESRI shapefiles.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

inventory = []
if LOCAL_DATA_DIR is None:
    api = HfApi()
    repo_files = api.list_repo_files(HF_REPO_ID, repo_type="dataset")
    put("noah_repo_total_files", len(repo_files))
else:
    repo_files = [str(p.relative_to(LOCAL_DATA_DIR)).replace(os.sep, "/")
                  for p in Path(LOCAL_DATA_DIR).rglob("*") if p.is_file()]

def match_aoi(path):
    name = path.lower().split("/")[-1]
    return name.endswith(".zip") and any(k in name for k in AOI_KEYWORDS)

wanted = []
for f in repo_files:
    low = f.lower()
    m = re.search(r"(?:^|/)flood/(5|25|100)yr/", low)
    if m and match_aoi(f) and f"{m.group(1)}yr" in RETURN_PERIODS:
        wanted.append(("flood", f"{m.group(1)}yr", f))
    m = re.search(r"stormsurgeadvisory\s*(\d)", low)
    if INCLUDE_STORM_SURGE and m and match_aoi(f):
        wanted.append(("storm_surge", f"SSA{m.group(1)}", f))
    if low.endswith(("metadata_flood.txt", "metadata_stormsurge.txt", "noah_license.pdf")):
        wanted.append(("metadata", "-", f))

assert any(w[0] == "flood" for w in wanted), "No Zambales/Olongapo flood ZIPs matched — inspect repo_files."
for w in wanted: print(w)

In [ ]:
local = {}
for kind, scen, f in wanted:
    if LOCAL_DATA_DIR is None:
        p = Path(hf_hub_download(HF_REPO_ID, f, repo_type="dataset", local_dir=RAW/"noah"))
    else:
        p = Path(LOCAL_DATA_DIR)/f
    local[(kind, scen, f)] = p
    inventory.append({"dataset": "Project NOAH (HF mirror)", "kind": kind, "scenario": scen,
                      "source_path": f, "url": f"https://huggingface.co/datasets/{HF_REPO_ID}/resolve/main/{f}",
                      "bytes": p.stat().st_size, "retrieved_utc": SUMMARY["run_utc"]})
pd.DataFrame(inventory)

In [ ]:
# Metadata text files shipped with NOAH: printed so the data dictionary can quote the official class definitions.
for (kind, scen, f), p in local.items():
    if kind == "metadata" and p.suffix == ".txt":
        print("="*80, "\n", f, "\n", "="*80)
        print(p.read_text(errors="replace")[:3000])

### 1.2 Reading the shapefiles
Each ZIP gets extracted, and every `.shp` inside is read and tagged with its hazard type and scenario. The CRS is read from the `.prj`. If a `.prj` is missing, the CRS is inferred from the coordinate range, and that inference is logged.

In [ ]:
def read_zip_layers(zpath, kind, scen):
    dest = INTERIM/"extracted"/kind/scen/zpath.stem
    if not dest.exists():
        with zipfile.ZipFile(zpath) as z: z.extractall(dest)
    frames = []
    for shp in sorted(dest.rglob("*.shp")):
        g = gpd.read_file(shp)
        crs_note = "from .prj"
        if g.crs is None:
            minx, miny, maxx, maxy = g.total_bounds
            if -180 <= minx <= 180 and -90 <= miny <= 90:
                g = g.set_crs("EPSG:4326"); crs_note = "inferred EPSG:4326 (degree range)"
            else:
                g = g.set_crs(METRIC_CRS); crs_note = f"inferred {METRIC_CRS} (metre range)"
        g["hazard_type"], g["scenario"], g["src_file"] = kind, scen, shp.name
        g.attrs["crs_note"] = crs_note
        frames.append((shp, g))
    return frames

raw_layers = {}           # (kind, scen) -> GeoDataFrame in original CRS
raw_profile = []
for (kind, scen, f), p in local.items():
    if kind == "metadata": continue
    for shp, g in read_zip_layers(p, kind, scen):
        key = (kind, scen)
        if key in raw_layers:
            if g.crs != raw_layers[key].crs: g = g.to_crs(raw_layers[key].crs)
            raw_layers[key] = pd.concat([raw_layers[key], g], ignore_index=True)
        else:
            raw_layers[key] = g
        raw_profile.append({"kind": kind, "scenario": scen, "zip": Path(f).name, "shp": shp.name,
                            "rows": len(g), "columns": ", ".join(c for c in g.columns if c != "geometry"),
                            "crs": str(g.crs), "crs_note": g.attrs.get("crs_note"),
                            "geom_types": ", ".join(sorted(g.geom_type.dropna().unique()))})
raw_profile = pd.DataFrame(raw_profile)
raw_profile.to_csv(OUT/"noah_raw_profile.csv", index=False)
put("noah_raw_profile", raw_profile.to_dict("records"))
raw_profile

### 1.3 Administrative boundaries (geoBoundaries gbOpen)
geoBoundaries ADM3 has no province attribute. So the area of interest (AOI) is built **spatially**: take the ADM2 polygons named Zambales or Olongapo, then keep the ADM3 units whose interior point falls inside them. The result is checked against the PSA list of 13 municipalities plus Olongapo City.

In [ ]:
def get_geoboundaries(lvl):
    fp = RAW/"boundaries"/f"geoBoundaries-PHL-{lvl}.geojson"
    meta_fp = RAW/"boundaries"/f"geoBoundaries-PHL-{lvl}.meta.json"
    if LOCAL_DATA_DIR is not None:
        fp = Path(LOCAL_DATA_DIR)/"boundaries"/fp.name
    if not fp.exists():
        fp.parent.mkdir(parents=True, exist_ok=True)
        meta = requests.get(GB_API.format(lvl=lvl), timeout=60).json()
        meta_fp.write_text(json.dumps(meta, indent=2))
        r = requests.get(meta["gjDownloadURL"], timeout=600); r.raise_for_status()
        fp.write_bytes(r.content)
        inventory.append({"dataset": "geoBoundaries gbOpen", "kind": "boundary", "scenario": lvl,
                          "source_path": meta.get("boundarySource"), "url": meta["gjDownloadURL"],
                          "bytes": fp.stat().st_size, "retrieved_utc": SUMMARY["run_utc"],
                          "license": meta.get("boundaryLicense"), "year_represented": meta.get("boundaryYearRepresented")})
    return gpd.read_file(fp)

adm2, adm3 = get_geoboundaries("ADM2"), get_geoboundaries("ADM3")
print(adm2.shape, adm3.shape, adm3.columns.tolist())

name_col = "shapeName"
prov = adm2[adm2[name_col].str.contains("|".join(AOI_KEYWORDS), case=False, na=False)]
print("ADM2 units matched:", prov[name_col].tolist())
assert len(prov), "No ADM2 unit matched AOI_KEYWORDS"
aoi_poly = prov.to_crs(METRIC_CRS).union_all() if hasattr(prov, "union_all") else prov.to_crs(METRIC_CRS).unary_union

adm3_m = adm3.to_crs(METRIC_CRS)
inside = adm3_m.representative_point().within(aoi_poly)
# Olongapo is a highly urbanised city; some boundary editions list it outside the province polygon.
inside |= adm3_m[name_col].str.contains("olongapo", case=False, na=False)
lgu = adm3_m[inside].copy()
lgu["lgu_name"] = (lgu[name_col].str.replace(r"\s*\(.*?\)", "", regex=True)
                                 .str.replace(r"(?i)^city of\s+", "", regex=True)
                                 .str.replace(r"(?i)\s+city$", "", regex=True).str.strip())
lgu["lgu_area_km2"] = lgu.area / 1e6
lgu = lgu[["shapeID", "lgu_name", "lgu_area_km2", "geometry"]].sort_values("lgu_name").reset_index(drop=True)

missing = sorted(set(EXPECTED_LGUS) - set(lgu.lgu_name))
extra   = sorted(set(lgu.lgu_name) - set(EXPECTED_LGUS))
put("aoi_lgu_count", len(lgu)); put("aoi_missing_vs_psa", missing); put("aoi_extra_vs_psa", extra)
put("aoi_area_km2", round(lgu.lgu_area_km2.sum(), 2))
lgu.drop(columns="geometry")

### 1.4 NASA POWER daily meteorology (one request per LGU)
POWER is a gridded satellite/reanalysis product (MERRA-2 plus satellite precipitation), not a rain gauge. Neighbouring LGUs can fall in the **same grid cell** and get identical series. The notebook detects this instead of treating them as independent observations.

In [ ]:
def fetch_power(lat, lon, name):
    fp = RAW/"nasa_power"/f"{name.replace(' ', '_')}.json"
    if LOCAL_DATA_DIR is not None: fp = Path(LOCAL_DATA_DIR)/"nasa_power"/fp.name
    if not fp.exists():
        fp.parent.mkdir(parents=True, exist_ok=True)
        q = {"parameters": ",".join(POWER_PARAMS), "community": "AG", "latitude": round(lat, 4),
             "longitude": round(lon, 4), "start": POWER_START, "end": POWER_END, "format": "JSON"}
        for attempt in range(4):
            r = requests.get(POWER_URL, params=q, timeout=180)
            if r.status_code == 200: break
            print(name, "HTTP", r.status_code, "retrying…"); time.sleep(5 * (attempt + 1))
        r.raise_for_status()
        fp.write_text(r.text); time.sleep(1)
        inventory.append({"dataset": "NASA POWER daily (AG)", "kind": "meteorology", "scenario": name,
                          "source_path": "temporal/daily/point", "url": r.url, "bytes": fp.stat().st_size,
                          "retrieved_utc": SUMMARY["run_utc"]})
    return json.loads(fp.read_text())

pts = lgu.set_geometry(lgu.representative_point()).to_crs(WEB_CRS)
rows, power_meta = [], {}
for _, r in pts.iterrows():
    js = fetch_power(r.geometry.y, r.geometry.x, r.lgu_name)
    fill = js.get("header", {}).get("fill_value", -999)
    par = js["properties"]["parameter"]
    df = pd.DataFrame(par); df.index.name = "date_str"; df = df.reset_index()
    df["lgu_name"], df["req_lat"], df["req_lon"] = r.lgu_name, r.geometry.y, r.geometry.x
    df["fill_value"] = fill
    rows.append(df)
    power_meta[r.lgu_name] = {"units": {k: v.get("units") for k, v in js.get("parameters", {}).items()},
                              "header_sources": js.get("header", {}).get("sources")}
power_raw = pd.concat(rows, ignore_index=True)
power_raw.to_csv(RAW/"nasa_power_raw_long.csv", index=False)
put("power_units", next(iter(power_meta.values()))["units"])
pd.DataFrame(inventory).to_csv(OUT/"data_inventory.csv", index=False)
print(power_raw.shape); power_raw.head()

## 2. Exploratory data analysis: raw (before preprocessing)
### 2.1 NOAH hazard polygons
Quality checks per layer: null/invalid hazard codes, invalid or empty geometries, exact duplicate geometries, sliver polygons, and area distribution.

In [ ]:
def hazard_field(g):
    for c in ["Var", "VAR", "var", "HAZ", "Haz", "haz", "GRIDCODE", "gridcode"]:
        if c in g.columns: return c
    raise KeyError(f"No hazard field in {g.columns.tolist()}")

eda_rows = []
for (kind, scen), g in raw_layers.items():
    hf = hazard_field(g)
    gm = g.to_crs(METRIC_CRS)
    area = gm.area
    vals = pd.to_numeric(g[hf], errors="coerce")
    wkb = g.geometry.apply(lambda x: x.wkb if x is not None else None)
    eda_rows.append({
        "kind": kind, "scenario": scen, "hazard_field": hf, "rows": len(g),
        "null_hazard": int(vals.isna().sum()),
        "invalid_hazard_code": int((~vals.isin([1, 2, 3]) & vals.notna()).sum()),
        "hazard_codes_seen": sorted(vals.dropna().unique().tolist()),
        "null_or_empty_geom": int((g.geometry.isna() | g.geometry.is_empty).sum()),
        "invalid_geom": int((~g.geometry.is_valid).sum()),
        "duplicate_geom": int(wkb.duplicated().sum()),
        "slivers_lt_1m2": int((area < SLIVER_M2).sum()),
        "area_km2_total": round(area.sum()/1e6, 3),
        "area_m2_median": round(float(area.median()), 1),
        "area_m2_p99": round(float(area.quantile(.99)), 1),
        "crs": str(g.crs)})
eda_noah_raw = pd.DataFrame(eda_rows)
eda_noah_raw.to_csv(OUT/"eda_noah_raw_quality.csv", index=False)
put("eda_noah_raw_quality", eda_noah_raw.to_dict("records"))
eda_noah_raw

In [ ]:
# Distribution of hazard classes (polygon count and area) per flood scenario — raw
fl = {s: raw_layers[("flood", s)] for s in RETURN_PERIODS if ("flood", s) in raw_layers}
recs = []
for s, g in fl.items():
    hf = hazard_field(g); a = g.to_crs(METRIC_CRS).area/1e6
    v = pd.to_numeric(g[hf], errors="coerce")
    for lvl in [1, 2, 3]:
        recs.append({"scenario": s, "level": HAZ_LABEL[lvl], "polygons": int((v == lvl).sum()),
                     "area_km2": float(a[v == lvl].sum())})
cls_raw = pd.DataFrame(recs)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for i, metric in enumerate(["polygons", "area_km2"]):
    sns.barplot(cls_raw, x="scenario", y=metric, hue="level", palette=SEQ3, ax=ax[i], order=list(fl))
    ax[i].set_title(f"Raw NOAH flood polygons: {'count' if i==0 else 'area (km²)'} by hazard level")
    ax[i].set_xlabel("Return period"); ax[i].legend(title="Hazard", frameon=False)
savefig("01_raw_flood_class_distribution")
put("raw_flood_class_distribution", cls_raw.to_dict("records"))

In [ ]:
# Polygon size distribution (log scale): shows slivers and the long tail of large floodplain polygons
fig, ax = plt.subplots(figsize=(7, 3.6))
for c, (s, g) in zip(SEQ3, fl.items()):
    a = g.to_crs(METRIC_CRS).area
    ax.hist(np.log10(a[a > 0]), bins=60, histtype="step", lw=2, color=c, label=s)
ax.axvline(np.log10(SLIVER_M2), color="#52514e", ls="--", lw=1); ax.text(np.log10(SLIVER_M2)+.05, ax.get_ylim()[1]*.9, "1 m² sliver cut", fontsize=8)
ax.set_xlabel("log10(polygon area, m²)"); ax.set_ylabel("Polygons"); ax.legend(frameon=False, title="Return period")
ax.set_title("Raw flood polygon size distribution")
savefig("02_raw_polygon_area_hist")

In [ ]:
# Map of the raw 100-yr layer over the AOI
s = "100yr" if "100yr" in fl else list(fl)[-1]
g = fl[s].to_crs(METRIC_CRS); hf = hazard_field(g)
fig, ax = plt.subplots(figsize=(6, 8))
lgu.boundary.plot(ax=ax, color="#8a8984", lw=.6)
for lvl, c in zip([1, 2, 3], SEQ3):
    sub = g[pd.to_numeric(g[hf], errors="coerce") == lvl]
    if len(sub): sub.plot(ax=ax, color=c, lw=0, label=HAZ_LABEL[lvl])
for _, r in lgu.iterrows():
    p = r.geometry.representative_point(); ax.annotate(r.lgu_name, (p.x, p.y), fontsize=6.5, ha="center", color="#0b0b0b")
ax.set_axis_off(); ax.set_title(f"NOAH flood hazard ({s}) — Zambales & Olongapo")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=HAZ_LABEL[l]) for l, c in zip([1,2,3], SEQ3)], frameon=False, loc="lower left", title="Hazard")
savefig("03_raw_flood_map_100yr")

### 2.2 NASA POWER meteorology

In [ ]:
pr = power_raw.copy()
pr["date"] = pd.to_datetime(pr["date_str"], format="%Y%m%d", errors="coerce")
fill_mask = pr[POWER_PARAMS].eq(pr["fill_value"].iloc[0])
miss = pd.DataFrame({"fill_value_count": fill_mask.sum(), "pct": (fill_mask.mean()*100).round(3)})
put("power_raw_shape", list(power_raw.shape))
put("power_raw_date_range", [str(pr.date.min().date()), str(pr.date.max().date())])
put("power_raw_missing", miss.to_dict("index"))
put("power_raw_bad_dates", int(pr.date.isna().sum()))
put("power_raw_duplicate_lgu_date", int(pr.duplicated(["lgu_name", "date_str"]).sum()))

# Where are the fill values? Trailing (processing lag) vs interior gaps behave differently.
def gap_profile(s, is_missing):
    last_valid = s[~is_missing].index.max()
    trailing = int(is_missing[s.index > last_valid].sum())
    interior = int(is_missing[s.index <= last_valid].sum())
    return trailing, interior
gp = []
for name, d in pr.set_index("date").groupby("lgu_name"):
    for p in POWER_PARAMS:
        t, i = gap_profile(d[p], d[p].eq(d["fill_value"].iloc[0]))
        gp.append({"lgu": name, "param": p, "trailing_missing": t, "interior_missing": i})
gp = pd.DataFrame(gp); gp.to_csv(OUT/"eda_power_gap_profile.csv", index=False)
put("power_gap_profile_totals", gp.groupby("param")[["trailing_missing", "interior_missing"]].sum().to_dict("index"))

# How many distinct grid series? (identical rainfall series ⇒ same POWER cell)
wide = pr.pivot_table(index="date", columns="lgu_name", values="PRECTOTCORR")
series_hash = wide.T.round(4).fillna(-1).apply(lambda r: hash(tuple(r.values)), axis=1)
cell_groups = series_hash.groupby(series_hash).apply(lambda s: sorted(s.index)).tolist()
put("power_distinct_rain_series", len(cell_groups)); put("power_cell_groups", cell_groups)
miss

In [ ]:
prv = pr.copy(); prv[POWER_PARAMS] = pr[POWER_PARAMS].mask(fill_mask)  # fill values → NaN for EDA only
desc = prv.groupby("lgu_name")[POWER_PARAMS].describe().round(2)
desc.to_csv(OUT/"eda_power_raw_describe.csv")
put("power_raw_describe_all", prv[POWER_PARAMS].describe().round(3).to_dict())
desc.xs("PRECTOTCORR", axis=1, level=0)

In [ ]:
one = cell_groups[0][0]
d = prv[prv.lgu_name == one].set_index("date")
fig, ax = plt.subplots(3, 1, figsize=(11, 8))
ax[0].plot(d.index, d.PRECTOTCORR, lw=.5, color=CAT[0]); ax[0].set_ylabel("mm/day")
ax[0].set_title(f"Daily rainfall, {one} grid cell (NASA POWER PRECTOTCORR)")
m = d.PRECTOTCORR.resample("MS").sum(min_count=20).to_frame("mm"); m["month"] = m.index.month
sns.boxplot(m, x="month", y="mm", color=SEQ3[0], ax=ax[1], fliersize=2, linewidth=.8)
ax[1].set_title("Monthly rainfall totals by calendar month (seasonality)"); ax[1].set_ylabel("mm/month")
yc = d.PRECTOTCORR.groupby(d.index.year).count()
amax = d.PRECTOTCORR.groupby(d.index.year).max()[yc >= 360]   # complete years only
ax[2].plot(amax.index, amax.values, marker="o", ms=4, lw=2, color=CAT[1]); ax[2].set_ylabel("mm/day")
ax[2].set_title("Annual maximum 1-day rainfall (complete years)")
savefig("04_raw_rainfall_overview")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
wet = prv.PRECTOTCORR[prv.PRECTOTCORR >= WET_DAY_MM]
ax[0].hist(wet, bins=80, color=CAT[0]); ax[0].set_yscale("log"); ax[0].set_xlabel("mm/day (wet days ≥ 1 mm)")
ax[0].set_title("Wet-day rainfall distribution (log count)")
sns.heatmap(prv[POWER_PARAMS].corr(),
            annot=True, fmt=".2f", cmap="Blues", vmin=-1, vmax=1, ax=ax[1], cbar=False)
ax[1].set_title("Correlation of daily POWER variables")
savefig("05_raw_rainfall_distribution_corr")
put("power_wet_day_share_pct", round(float((prv.PRECTOTCORR >= WET_DAY_MM).mean()*100), 2))

## 3. Data cleaning
**NOAH polygons:**
- (a) Reproject to the metric CRS.
- (b) Repair invalid geometries with `make_valid` and keep only polygonal parts.
- (c) Drop null/empty geometries.
- (d) Drop rows whose hazard code is not 1/2/3. A hazard class can't be imputed from neighbours without re-running the hydraulic model, so dropping is the only honest option.
- (e) Drop exact duplicate geometries.
- (f) Drop slivers smaller than 1 m².
- (g) Keep only standardised columns.

**POWER:** fill value → NaN, parse dates, de-duplicate `(lgu, date)`, and range-check values against physical limits.

In [ ]:
def polygonal(geom):
    if geom is None or geom.is_empty: return None
    if not geom.is_valid: geom = make_valid(geom)
    if geom.geom_type in ("Polygon", "MultiPolygon"): return geom
    if geom.geom_type == "GeometryCollection":
        parts = [p for p in geom.geoms if p.geom_type in ("Polygon", "MultiPolygon")]
        if not parts: return None
        from shapely.ops import unary_union; return unary_union(parts)
    return None

clean_layers, clean_log = {}, []
for (kind, scen), g in raw_layers.items():
    hf = hazard_field(g); n0 = len(g); log = {"kind": kind, "scenario": scen, "rows_in": n0}
    g = g.to_crs(METRIC_CRS)
    g["hazard_level"] = pd.to_numeric(g[hf], errors="coerce")
    bad = ~g.hazard_level.isin([1, 2, 3]); log["drop_bad_hazard"] = int(bad.sum()); g = g[~bad]
    inval = int((~g.geometry.is_valid).sum())
    g = g.set_geometry(gpd.GeoSeries(g.geometry.apply(polygonal), index=g.index, crs=g.crs)); log["repaired_invalid"] = inval
    empty = g.geometry.isna() | g.geometry.is_empty; log["drop_empty"] = int(empty.sum()); g = g[~empty]
    g = g.explode(index_parts=False, ignore_index=True)
    dup = g.geometry.apply(lambda x: x.wkb).duplicated() ; log["drop_duplicate_geom"] = int(dup.sum()); g = g[~dup]
    sl = g.area < SLIVER_M2; log["drop_slivers"] = int(sl.sum()); g = g[~sl]
    g["hazard_level"] = g.hazard_level.astype("int8")
    g["hazard_label"] = g.hazard_level.map(HAZ_LABEL)
    g = g[["hazard_type", "scenario", "hazard_level", "hazard_label", "geometry"]].reset_index(drop=True)
    log["rows_out"] = len(g); log["area_km2_out"] = round(g.area.sum()/1e6, 3)
    clean_layers[(kind, scen)] = g; clean_log.append(log)
clean_log = pd.DataFrame(clean_log); clean_log.to_csv(OUT/"cleaning_log_noah.csv", index=False)
put("cleaning_log_noah", clean_log.to_dict("records")); clean_log

In [ ]:
pw = power_raw.copy()
pw["date"] = pd.to_datetime(pw["date_str"], format="%Y%m%d", errors="coerce")
n0 = len(pw)
pw = pw.dropna(subset=["date"]).drop_duplicates(["lgu_name", "date"])
for p in POWER_PARAMS:
    pw[p] = pd.to_numeric(pw[p], errors="coerce").mask(lambda s: s == pw["fill_value"].iloc[0])
range_flags = {
    "PRECTOTCORR_negative": int((pw.PRECTOTCORR < 0).sum()),
    "PRECTOTCORR_gt_plausible": int((pw.PRECTOTCORR > RAIN_MAX_PLAUSIBLE).sum()),
    "T2M_out_10_45C": int(((pw.T2M < 10) | (pw.T2M > 45)).sum()),
    "RH2M_out_0_100": int(((pw.RH2M < 0) | (pw.RH2M > 100)).sum())}
pw.loc[pw.PRECTOTCORR < 0, "PRECTOTCORR"] = np.nan          # physically impossible → missing
pw.loc[(pw.RH2M < 0) | (pw.RH2M > 100), "RH2M"] = np.nan
# Implausibly large rain is FLAGGED, not deleted: extremes are the signal for flood work.
pw["flag_rain_extreme"] = pw.PRECTOTCORR > RAIN_MAX_PLAUSIBLE
put("power_range_flags", range_flags); put("power_rows_dropped_dup_or_baddate", n0 - len(pw))

# Trailing gap = POWER processing lag, not missing data. Trim to the last date where ALL LGUs have rainfall.
last_complete = pw.dropna(subset=["PRECTOTCORR"]).groupby("lgu_name").date.max().min()
pw = pw[pw.date <= last_complete].drop(columns=["date_str", "fill_value"])
put("power_clean_end_date", str(last_complete.date()))
pw.isna().sum()

## 4. Missing-value imputation
Methods are chosen **per variable**, based on how each one behaves:

| Variable | Behaviour | Method | Why |
|---|---|---|---|
| NOAH `hazard_level` | model output, categorical | **none (drop)** | Inventing a hazard class would put fake risk on a map people use for safety. |
| `T2M`, `RH2M` | smooth, strongly autocorrelated | time interpolation, gaps ≤ 3 days | Neighbouring days carry most of the information. Longer gaps stay NaN. |
| `PRECTOTCORR` | intermittent, zero-inflated, heavy-tailed | **spatial donor**: same-day value from the nearest *distinct* POWER cell; else stays NaN | Linear interpolation smears rain across dry days and invents events. Mean/median fills are worse, because the median is ~0 mm. |

Every imputed cell is flagged (`imp_*` columns), so later modelling can exclude them.

In [ ]:
pw = pw.sort_values(["lgu_name", "date"]).reset_index(drop=True)
before = pw[POWER_PARAMS].isna().sum()

# Smooth variables: time interpolation within each LGU, gaps ≤ MAX_INTERP_DAYS only
for p in ["T2M", "RH2M"]:
    pw[f"imp_{p}"] = False
    for name, idx in pw.groupby("lgu_name").groups.items():
        s = pw.loc[idx, ["date", p]].set_index("date")[p]
        isna = s.isna()
        run_id = (isna != isna.shift()).cumsum()
        run_len = isna.groupby(run_id).transform("sum")
        fillable = isna & (run_len <= MAX_INTERP_DAYS)
        filled = s.interpolate(method="time", limit_area="inside")
        s2 = s.where(~fillable, filled)
        pw.loc[idx, p] = s2.values; pw.loc[idx, f"imp_{p}"] = (fillable & s2.notna()).values

# Rainfall: spatial donor from the nearest DISTINCT grid series
rain_w = pw.pivot(index="date", columns="lgu_name", values="PRECTOTCORR")
cent = lgu.set_index("lgu_name").geometry.representative_point()
rep = {n: grp[0] for grp in cell_groups for n in grp}          # LGU → representative of its cell
pw["imp_PRECTOTCORR"] = False
for name in rain_w.columns:
    mask = rain_w[name].isna()
    if not mask.any(): continue
    donors = [g[0] for g in cell_groups if rep[name] not in g]
    donors = sorted(donors, key=lambda d: cent[name].distance(cent[d]))
    for d in donors:
        fill_here = mask & rain_w[d].notna()
        rain_w.loc[fill_here, name] = rain_w.loc[fill_here, d]
        sel = (pw.lgu_name == name) & pw.date.isin(rain_w.index[fill_here])
        pw.loc[sel, "PRECTOTCORR"] = rain_w.loc[fill_here, name].values
        pw.loc[sel, "imp_PRECTOTCORR"] = True
        mask = rain_w[name].isna()
        if not mask.any(): break

after = pw[POWER_PARAMS].isna().sum()
imp = pd.DataFrame({"missing_before": before, "imputed": [int(pw[f"imp_{p}"].sum()) for p in POWER_PARAMS],
                    "still_missing": after})
put("imputation_summary", imp.to_dict("index")); imp

## 5. Data integration
Three sources, three reference systems. They are reconciled like this:
- **CRS:** NOAH (`.prj`), geoBoundaries (EPSG:4326), and POWER points (lat/lon) all go to EPSG:32651 for any area math.
- **Spatial unit:** NOAH polygons are intersected with the LGU boundaries, giving hazard area per LGU, scenario, and level.
- **Naming:** geoBoundaries names are normalised (strip "City of"/"City"). POWER series are keyed by the same `lgu_name`.
- **Units:** areas in km² and shares in %. Rainfall stays in mm/day (POWER native).
- **Time:** POWER dates are parsed from `YYYYMMDD` (local solar time) to `datetime64`. NOAH has no time axis, so it joins on space only.

In [ ]:
lgu_slim = lgu[["lgu_name", "lgu_area_km2", "geometry"]]
by_lgu = []
for (kind, scen), g in clean_layers.items():
    ov = gpd.overlay(g, lgu_slim, how="intersection", keep_geom_type=True)
    ov = ov.dissolve(by=["lgu_name", "hazard_type", "scenario", "hazard_level", "hazard_label"], as_index=False)
    ov["area_km2"] = ov.area / 1e6
    by_lgu.append(ov)
    inside_km2 = ov.area_km2.sum(); total_km2 = g.area.sum()/1e6
    SUMMARY.setdefault("integration_outside_aoi_km2", {})[f"{kind}_{scen}"] = round(total_km2 - inside_km2, 3)
hz = pd.concat(by_lgu, ignore_index=True)
hz = gpd.GeoDataFrame(hz, geometry="geometry", crs=METRIC_CRS)
hz["share_of_lgu_pct"] = hz.area_km2 / hz.lgu_area_km2 * 100

# Quality check: do different hazard levels overlap inside one scenario? (they should not)
ovl = []
for (n, k, s), grp in hz.groupby(["lgu_name", "hazard_type", "scenario"]):
    u = grp.union_all() if hasattr(grp, "union_all") else grp.unary_union
    ovl.append(grp.area.sum()/1e6 - u.area/1e6)
put("integration_class_overlap_km2_max", round(float(np.max(ovl)), 4))
print(SUMMARY["integration_outside_aoi_km2"])
hz.drop(columns="geometry").head()

In [ ]:
# Wide LGU feature table: hazard shares per scenario/level
wide = (hz.assign(col=lambda d: d.hazard_type.str.replace("storm_surge", "ss") + "_" + d.scenario.str.lower()
                               + "_" + d.hazard_label.str.lower() + "_pct")
          .pivot_table(index="lgu_name", columns="col", values="share_of_lgu_pct", aggfunc="sum").fillna(0.0))
# Missing here genuinely means 0: no polygon of that class intersects the LGU (structural zero, not unknown).
for s in RETURN_PERIODS:
    cols = [c for c in wide if c.startswith(f"flood_{s}_")]
    if cols: wide[f"flood_{s}_any_pct"] = wide[cols].sum(axis=1)

# Rainfall climatology per LGU, from the cleaned daily data
pwi = pw.set_index("date")
def rain_stats(d):
    yrs = d.PRECTOTCORR.groupby(d.index.year)
    full = yrs.count() >= 360
    return pd.Series({
        "rain_mean_annual_mm": yrs.sum()[full].mean(),
        "rain_mean_annual_max1d_mm": yrs.max()[full].mean(),
        "rain_p95_wetday_mm": d.PRECTOTCORR[d.PRECTOTCORR >= WET_DAY_MM].quantile(.95),
        "rain_wetdays_per_year": (d.PRECTOTCORR >= WET_DAY_MM).groupby(d.index.year).sum()[full].mean(),
        "rain_wetseason_share_pct": d.PRECTOTCORR[d.index.month.isin(WET_SEASON_MONTHS)].sum() / d.PRECTOTCORR.sum() * 100,
        "t2m_mean_c": d.T2M.mean(), "rh2m_mean_pct": d.RH2M.mean()})
rain_lgu = pwi.groupby("lgu_name").apply(rain_stats)

cent_ll = lgu.set_geometry(lgu.representative_point()).to_crs(WEB_CRS)
geo = pd.DataFrame({"lgu_name": cent_ll.lgu_name, "centroid_lat": cent_ll.geometry.y, "centroid_lon": cent_ll.geometry.x,
                    "lgu_area_km2": lgu.lgu_area_km2, "power_cell_group": [rep[n] for n in cent_ll.lgu_name]}).set_index("lgu_name")
lgu_features = geo.join(wide, how="left").join(rain_lgu, how="left").fillna({c: 0.0 for c in wide.columns}).reset_index()
lgu_features.to_csv(INTERIM/"lgu_features_full.csv", index=False)
put("lgu_features_shape", list(lgu_features.shape)); lgu_features.round(2)

## 6. Data transformation
- **Date/time features** (daily table): year, month, day-of-year, `season` (wet May–Oct / dry Nov–Apr, Coronas Type I), cyclical month encoding (sin/cos), and wet-day flag.
- **Antecedent rainfall:** 3-, 7-, and 30-day rolling totals. Floods respond to accumulated rain, not just one day. Windows that touch a still-missing day stay NaN (`min_periods` = window).
- **Extreme-day flag:** rainfall ≥ the LGU's own 95th-percentile wet day.
- **Categorical:** `hazard_level` stays an ordinal int (1–3) plus a one-hot copy. `season` is one-hot encoded.
- **Scaling:** z-score standardisation of numeric LGU features (for later clustering/modelling). Raw values are kept alongside for the map.
- **Spatial:** centroids in WGS84 lat/lon and web layers reprojected to EPSG:4326.

In [ ]:
pw = pw.sort_values(["lgu_name", "date"]).reset_index(drop=True)
pw["year"], pw["month"], pw["doy"] = pw.date.dt.year, pw.date.dt.month, pw.date.dt.dayofyear
pw["season"] = np.where(pw.month.isin(WET_SEASON_MONTHS), "wet", "dry")
pw["month_sin"], pw["month_cos"] = np.sin(2*np.pi*pw.month/12), np.cos(2*np.pi*pw.month/12)
pw["is_wet_day"] = (pw.PRECTOTCORR >= WET_DAY_MM).astype("Int8")
for w in [3, 7, 30]:
    pw[f"rain_{w}d_mm"] = pw.groupby("lgu_name").PRECTOTCORR.transform(lambda s: s.rolling(w, min_periods=w).sum())
p95 = pw[pw.PRECTOTCORR >= WET_DAY_MM].groupby("lgu_name").PRECTOTCORR.quantile(.95)
pw["is_extreme_day"] = (pw.PRECTOTCORR >= pw.lgu_name.map(p95)).astype("Int8")
pw = pd.concat([pw, pd.get_dummies(pw.season, prefix="season", dtype="int8")], axis=1)

num_cols = [c for c in lgu_features.columns if c not in ("lgu_name", "power_cell_group")]
scaled = lgu_features.copy()
for c in num_cols:
    sd = scaled[c].std(ddof=0)
    scaled[c] = 0.0 if sd == 0 or np.isnan(sd) else (scaled[c] - scaled[c].mean())/sd
put("transform_zero_variance_cols", [c for c in num_cols if lgu_features[c].std(ddof=0) == 0])
pw.head()

## 7. Data reduction
- **Rows/geometry:** polygons are dissolved by LGU × scenario × level (done in §5), cutting tens of thousands of polygons to a few hundred features. Web layers are simplified at 5 m tolerance, and the vertex reduction is reported.
- **Columns:** NOAH bookkeeping fields are dropped. In the LGU feature table, any feature with |r| > 0.95 to an earlier-listed feature is dropped as redundant, and the correlated pairs are logged.
- **Why not PCA:** with ~14 LGUs and a map-first product, principal components would make the features uninterpretable for residents and LGU officers. Correlation filtering keeps the original, explainable variables.

In [ ]:
X = lgu_features[num_cols].loc[:, lgu_features[num_cols].std(ddof=0) > 0]
corr = X.corr().abs()
drop, pairs = set(), []
cols = list(X.columns)
for i, a in enumerate(cols):
    for b in cols[i+1:]:
        if a not in drop and b not in drop and corr.loc[a, b] > CORR_DROP and not b.startswith(("centroid", "lgu_area")):
            drop.add(b); pairs.append((a, b, round(float(corr.loc[a, b]), 3)))
keep_cols = ["lgu_name", "power_cell_group"] + [c for c in num_cols if c not in drop]
put("reduction_dropped_redundant", sorted(drop)); put("reduction_corr_pairs", pairs)
lgu_reduced, lgu_scaled_reduced = lgu_features[keep_cols], scaled[keep_cols]

plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, cbar_kws={"shrink": .6})
plt.title("LGU feature correlation (before reduction)")
savefig("06_lgu_feature_correlation")

def nverts(gs):
    def one(g):
        if g is None or g.is_empty or g.geom_type not in ("Polygon", "MultiPolygon"): return 0
        return sum(len(p.exterior.coords) + sum(len(i.coords) for i in p.interiors)
                   for p in (g.geoms if g.geom_type == "MultiPolygon" else [g]))
    return int(gs.apply(one).sum())
v_raw = sum(nverts(g.geometry) for g in raw_layers.values())
hz_web = hz.copy(); hz_web["geometry"] = hz_web.geometry.simplify(SIMPLIFY_TOL_M, preserve_topology=True)
put("reduction_features_raw_vs_dissolved", [int(sum(len(g) for g in raw_layers.values())), int(len(hz))])
put("reduction_vertices_raw_vs_web", [v_raw, nverts(hz_web.geometry)])

## 8. Before vs. after

In [ ]:
order = {f"flood_{s}": i for i, s in enumerate(RETURN_PERIODS)}
ba = clean_log[["kind", "scenario", "rows_in", "rows_out"]].copy()
ba = ba.iloc[sorted(range(len(ba)), key=lambda i: (order.get(f"{ba.kind.iloc[i]}_{ba.scenario.iloc[i]}", 99), ba.scenario.iloc[i]))].reset_index(drop=True)
ba["features_after_dissolve"] = [int(((hz.hazard_type == k) & (hz.scenario == s)).sum()) for k, s in zip(ba.kind, ba.scenario)]
ba.to_csv(OUT/"before_after_noah.csv", index=False); put("before_after_noah", ba.to_dict("records"))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
lab = ba.kind.str.replace("storm_surge", "SS").str.replace("flood", "Flood") + " " + ba.scenario
x = np.arange(len(ba)); w = .27
for i, (col, c) in enumerate(zip(["rows_in", "rows_out", "features_after_dissolve"], SEQ3)):
    ax[0].bar(x + (i-1)*w, ba[col], w, color=c, label=col.replace("_", " "))
ax[0].set_yscale("log"); ax[0].set_xticks(x, lab, rotation=45, ha="right")
ax[0].legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(.5, -.32))
ax[0].set_title("NOAH features: raw → cleaned → dissolved (log)")
miss_b = before.rename("before"); miss_a = after.rename("after")
pd.concat([miss_b, miss_a], axis=1).plot.bar(ax=ax[1], color=[SEQ3[0], SEQ3[2]], rot=0)
ax[1].set_title("POWER interior missing days: before vs after imputation"); ax[1].legend(frameon=False)
savefig("07_before_after")

In [ ]:
# Hazard share per LGU (cleaned + integrated): the core layer for DIA
fcols = [f"flood_{s}_high_pct" for s in RETURN_PERIODS if f"flood_{s}_high_pct" in lgu_features]
t = lgu_features.set_index("lgu_name")[fcols].sort_values(fcols[-1])
t.columns = [c.split("_")[1] for c in fcols]
ax = t.plot.barh(figsize=(8, 6), color=SEQ3[:len(fcols)], width=.8)
ax.set_xlabel("% of LGU land area in HIGH flood hazard (>1.5 m)"); ax.set_ylabel("")
ax.set_title("High flood hazard share by LGU and return period"); ax.legend(title="Return period", frameon=False)
savefig("08_lgu_high_hazard_share")

## 9. Hypothesis tests (Phase 1, descriptive/inferential)
- **H1:** The share of LGU area in **high** flood hazard is larger under the 100-yr scenario than the 5-yr one. One-sided Wilcoxon signed-rank test, paired by LGU.
- **H2:** Monthly rainfall in the wet season (May–Oct) is greater than in the dry season (Nov–Apr). One-sided Mann–Whitney U on regional monthly totals (mean of distinct POWER cells).
- **H3:** Annual maximum 1-day rainfall shows a monotonic trend over 1981–present. Mann–Kendall via Kendall's τ against year, per distinct cell.
- **H4:** LGUs with more 100-yr flood-hazard area also have more storm-surge (SSA4) hazard area. Spearman ρ.

With n ≈ 14 LGUs (and fewer independent rainfall cells), these are exploratory. Effect sizes are reported next to p-values.

In [ ]:
H = {}
lf = lgu_features.set_index("lgu_name")
if {"flood_5yr_high_pct", "flood_100yr_high_pct"} <= set(lf):
    a, b = lf.flood_5yr_high_pct, lf.flood_100yr_high_pct
    res = stats.wilcoxon(b, a, alternative="greater", zero_method="wilcox") if (b - a).abs().sum() > 0 else None
    H["H1"] = {"n_lgu": len(lf), "median_5yr": a.median(), "median_100yr": b.median(),
               "median_diff_pp": (b - a).median(), "lgus_increased": int((b > a).sum()),
               "W": None if res is None else res.statistic, "p_one_sided": None if res is None else res.pvalue}

reps = [g[0] for g in cell_groups]
# Regional series = mean of the DISTINCT POWER cells (avoids pseudo-replication from spatially correlated cells)
reg = pw[pw.lgu_name.isin(reps)].pivot(index="date", columns="lgu_name", values="PRECTOTCORR").mean(axis=1)
mon = reg.resample("MS").sum(min_count=25).dropna().rename("PRECTOTCORR").reset_index()
mon.columns = ["date", "PRECTOTCORR"]
mon["season"] = np.where(mon.date.dt.month.isin(WET_SEASON_MONTHS), "wet", "dry")
u = stats.mannwhitneyu(mon[mon.season == "wet"].PRECTOTCORR, mon[mon.season == "dry"].PRECTOTCORR, alternative="greater")
H["H2"] = {"n_months_wet": int((mon.season == "wet").sum()), "n_months_dry": int((mon.season == "dry").sum()),
           "median_wet_mm": mon[mon.season == "wet"].PRECTOTCORR.median(), "median_dry_mm": mon[mon.season == "dry"].PRECTOTCORR.median(),
           "U": u.statistic, "p_one_sided": u.pvalue}
# rank-biserial r = 2U/(n1 n2) - 1  (positive ⇒ wet > dry)
H["H2"]["rank_biserial"] = 2*u.statistic/(H["H2"]["n_months_wet"]*H["H2"]["n_months_dry"]) - 1

H["H3"] = {}
for r_ in reps:
    d = pw[pw.lgu_name == r_].set_index("date").PRECTOTCORR
    yrs = d.groupby(d.index.year); amax = yrs.max()[yrs.count() >= 360]
    tau = stats.kendalltau(amax.index, amax.values)
    slope = stats.theilslopes(amax.values, amax.index)
    H["H3"][r_] = {"years": len(amax), "tau": tau.statistic, "p_two_sided": tau.pvalue, "sen_slope_mm_per_decade": slope[0]*10}

if "flood_100yr_any_pct" in lf and any(c.startswith("ss_ssa4") for c in lf):
    ss = lf[[c for c in lf if c.startswith("ss_ssa4")]].sum(axis=1)
    rho = stats.spearmanr(lf.flood_100yr_any_pct, ss)
    H["H4"] = {"n_lgu": len(lf), "rho": rho.statistic, "p_two_sided": rho.pvalue,
               "lgus_with_ss4": int((ss > 0).sum())}
put("hypothesis_tests", H)

## 10. Export
| File | Contents | Use in DIA |
|---|---|---|
| `clean/dia_hazard_by_lgu.gpkg` | dissolved hazard polygons per LGU/scenario/level (EPSG:32651) | analysis |
| `clean/dia_hazard_web_<scenario>.geojson` | simplified, EPSG:4326 | map layers |
| `clean/dia_lgu_boundaries.geojson` | AOI LGU polygons, EPSG:4326 | map base / joins |
| `clean/lgu_features.csv`, `lgu_features_scaled.csv` | reduced LGU feature table (raw & z-scored) | Phase 2 modelling |
| `clean/rainfall_daily_clean.csv` | daily POWER + engineered features + imputation flags | time-series analysis |
| `data_dictionary.csv` | every exported column | report §Data Dictionary |
| `run_summary.json` | every statistic quoted in the report | report |

In [ ]:
hz.to_file(CLEAN/"dia_hazard_by_lgu.gpkg", layer="hazard_by_lgu", driver="GPKG")
hz_web_ll = hz_web.to_crs(WEB_CRS)
for (k, s), grp in hz_web_ll.groupby(["hazard_type", "scenario"]):
    grp.drop(columns=["lgu_area_km2"]).to_file(CLEAN/f"dia_hazard_web_{k}_{s}.geojson", driver="GeoJSON")
lgu.to_crs(WEB_CRS).to_file(CLEAN/"dia_lgu_boundaries.geojson", driver="GeoJSON")
lgu_reduced.round(4).to_csv(CLEAN/"lgu_features.csv", index=False)
lgu_scaled_reduced.round(4).to_csv(CLEAN/"lgu_features_scaled.csv", index=False)
pw_out = pw.drop(columns=["req_lat", "req_lon"])
pw_out.to_csv(CLEAN/"rainfall_daily_clean.csv", index=False)
pd.DataFrame(inventory).to_csv(OUT/"data_inventory.csv", index=False)
sorted(p.name for p in CLEAN.iterdir())

In [ ]:
# ---- Data dictionary: descriptions written here; dtype/range observed from the exported data ----
DESC = {
 "hazard_type": ("NOAH hazard family (flood or storm_surge)", "-", "Layer identity"),
 "scenario": ("Flood return period (5yr/25yr/100yr) or Storm Surge Advisory level (SSA1–SSA4)", "-", "Scenario selector on the map"),
 "hazard_level": ("NOAH hazard class (flood `Var` / storm surge `HAZ`): 1 Low, 2 Medium, 3 High", "ordinal", "Target/colour class"),
 "hazard_label": ("Text label of hazard_level", "-", "Legend"),
 "lgu_name": ("Municipality/city name (geoBoundaries ADM3, normalised)", "-", "Join key"),
 "lgu_area_km2": ("LGU land area computed in EPSG:32651", "km²", "Denominator for shares"),
 "area_km2": ("Area of the dissolved hazard polygon inside the LGU", "km²", "Exposure magnitude"),
 "share_of_lgu_pct": ("area_km2 / lgu_area_km2 × 100", "%", "Comparable exposure across LGUs"),
 "centroid_lat": ("Interior point latitude (WGS84)", "degrees", "Map label / POWER query point"),
 "centroid_lon": ("Interior point longitude (WGS84)", "degrees", "Map label / POWER query point"),
 "power_cell_group": ("Representative LGU of the POWER grid cell (LGUs sharing a cell share rainfall)", "-", "Avoid double-counting rainfall"),
 "date": ("Calendar day (POWER local solar time)", "date", "Time key"),
 "PRECTOTCORR": ("Bias-corrected total precipitation (NASA POWER)", "mm/day", "Primary flood driver"),
 "T2M": ("Air temperature at 2 m", "°C", "Context / seasonality"),
 "RH2M": ("Relative humidity at 2 m", "%", "Context / seasonality"),
 "flag_rain_extreme": ("Rainfall above plausibility ceiling (kept, flagged)", "bool", "Quality flag"),
 "imp_PRECTOTCORR": ("Rainfall value imputed from nearest distinct POWER cell", "bool", "Exclude in sensitivity checks"),
 "imp_T2M": ("T2M imputed by time interpolation (gap ≤3 d)", "bool", "Quality flag"),
 "imp_RH2M": ("RH2M imputed by time interpolation (gap ≤3 d)", "bool", "Quality flag"),
 "year": ("Calendar year", "-", "Trend analysis"), "month": ("Calendar month 1–12", "-", "Seasonality"),
 "doy": ("Day of year 1–366", "-", "Seasonality"), "season": ("wet (May–Oct) / dry (Nov–Apr), Coronas Type I", "-", "Seasonality"),
 "month_sin": ("sin(2π·month/12)", "-", "Cyclical encoding"), "month_cos": ("cos(2π·month/12)", "-", "Cyclical encoding"),
 "is_wet_day": ("PRECTOTCORR ≥ 1 mm", "0/1", "Rain frequency"),
 "rain_3d_mm": ("3-day antecedent rainfall total", "mm", "Soil saturation proxy"),
 "rain_7d_mm": ("7-day antecedent rainfall total", "mm", "Soil saturation proxy"),
 "rain_30d_mm": ("30-day antecedent rainfall total", "mm", "Seasonal wetness"),
 "is_extreme_day": ("Rain ≥ LGU's 95th percentile wet-day amount", "0/1", "Flood-trigger candidate"),
 "season_dry": ("One-hot of season", "0/1", "Model input"), "season_wet": ("One-hot of season", "0/1", "Model input"),
 "rain_mean_annual_mm": ("Mean annual rainfall (complete years)", "mm/yr", "Climatology"),
 "rain_mean_annual_max1d_mm": ("Mean of annual maximum 1-day rainfall", "mm", "Extreme-rain exposure"),
 "rain_p95_wetday_mm": ("95th percentile of wet-day rainfall", "mm/day", "Extreme threshold"),
 "rain_wetdays_per_year": ("Mean wet days (≥1 mm) per year", "days", "Rain frequency"),
 "rain_wetseason_share_pct": ("Share of rainfall falling May–Oct", "%", "Seasonality"),
 "t2m_mean_c": ("Mean daily air temperature", "°C", "Context"), "rh2m_mean_pct": ("Mean relative humidity", "%", "Context"),
}
def describe_col(c, s, table):
    base = DESC.get(c)
    if base is None and c.endswith("_pct"):
        parts = c[:-4].split("_")
        base = (f"% of LGU area in {parts[0].replace('ss','storm surge')} {parts[1].upper()} "
                f"{parts[2]} hazard" + (" (all levels)" if parts[2] == "any" else ""), "%", "Exposure feature")
    base = base or ("", "", "")
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        rng = f"{s.min():.4g} – {s.max():.4g}" if s.notna().any() else "all missing"
    else:
        u = s.dropna().astype(str).unique(); rng = ", ".join(sorted(u)[:8]) + (" …" if len(u) > 8 else "")
    return {"table": table, "feature": c, "description": base[0], "dtype": str(s.dtype), "unit": base[1],
            "values_or_range": rng, "missing": int(s.isna().sum()), "role": base[2]}
dd = []
for tname, df in [("dia_hazard_by_lgu", hz.drop(columns="geometry")), ("lgu_features", lgu_reduced),
                  ("rainfall_daily_clean", pw_out)]:
    dd += [describe_col(c, df[c], tname) for c in df.columns]
dd = pd.DataFrame(dd); dd.to_csv(OUT/"data_dictionary.csv", index=False)
print(len(dd), "dictionary rows"); dd.head(20)

In [ ]:
SUMMARY["inventory"] = pd.DataFrame(inventory).to_dict("records")
(OUT/"run_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=str))
shutil.make_archive("FSAI_DIA_Phase1_outputs", "zip", OUT)
print("Wrote", OUT/"run_summary.json", "and FSAI_DIA_Phase1_outputs.zip")
try:
    from google.colab import files; files.download("FSAI_DIA_Phase1_outputs.zip")
except Exception:
    pass